# Plan de experimentos en GPU (Kaggle)

Ejecuta con el orquestador del proyecto (`plan_experimentos.py`) dos planes seguidos:

1. **`planes/plan_prueba.json`**: el plan completo en miniatura (`--quick`), unos minutos. Si algo falla en GPU, sale aquí.
2. **`planes/plan_kaggle.json`**: todos los experimentos (variantes principales, ablación de la fracción, ER-ACE, cRT,
   variantes de WA, referencia conjunta x2, calendario de replay y reparto Dirichlet) **solo con la semilla 42**,
   para comparar después con la misma semilla en otra GPU. Unas 7-8 horas.

Las tareas se reparten entre las dos GPU. Al final se muestra el informe (`INFORME.md`) y todo queda en
`resultados_kaggle.tar.gz` (pestaña *Output*).

**Antes de ejecutar** (panel *Session options*): **Accelerator → GPU T4 x2** e **Internet → On**.
Para que siga aunque cierres el navegador: *Save Version* → *Save & Run All (Commit)*.

In [ ]:
# ---------------------------------------------------------------- parámetros
PLANES = ["planes/plan_prueba.json", "planes/plan_kaggle.json"]   # en este orden; quita el primero si ya lo pasaste
COMMIT = "6ceab93"           # revisión del código en GitHub

In [ ]:
!nvidia-smi -L

In [ ]:
%%bash -s "$COMMIT"
set -e
cd /kaggle/working
rm -rf cfl-practicas
git clone -q https://github.com/Esteban7754/cfl-practicas.git
cd cfl-practicas && git checkout -q "$1" && git log -1 --oneline

## Entorno: mismas versiones fijadas que la imagen Docker, con PyTorch CUDA

In [ ]:
%%bash
set -e
cd /kaggle/working/cfl-practicas
pip install -q uv
uv venv -q -p 3.13 /tmp/venv
sed -e '/^--extra-index-url/d' -e 's/+cpu//' requirements.lock > /tmp/req-cuda.txt
uv pip install -q -p /tmp/venv/bin/python -r /tmp/req-cuda.txt

ok() { /tmp/venv/bin/python -c "import torch; x = torch.ones(8, device='cuda'); assert float((x * 2).sum()) == 16" 2>/dev/null; }
if ! ok; then
    TORCH=$(grep '^torch==' /tmp/req-cuda.txt); VISION=$(grep '^torchvision==' /tmp/req-cuda.txt)
    for cu in cu130 cu128 cu126; do
        echo "PyTorch de PyPI no usa la GPU; probando $TORCH con $cu"
        uv pip install -q -p /tmp/venv/bin/python --reinstall "$TORCH" "$VISION" \
            --index-url "https://download.pytorch.org/whl/$cu" && ok && break || true
    done
fi
ok || { echo "[ERROR] PyTorch no puede usar la GPU. ¿Accelerator = GPU T4 x2?"; exit 1; }
/tmp/venv/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

In [ ]:
import os, subprocess
os.environ.pop("PYTHONPATH", None)   # sin el sitecustomize de Kaggle
os.environ.update(HF_HOME="/tmp/hf", MPLBACKEND="Agg", PYTHONHASHSEED="0", PYTHONUNBUFFERED="1",
                  CUBLAS_WORKSPACE_CONFIG=":4096:8")
REPO = "/kaggle/working/cfl-practicas"
PY = "/tmp/venv/bin/python"

# CIFAR-100 en la revisión fijada; después todo trabaja sin red.
!cd {REPO} && {PY} docker/descargar_datos.py
os.environ["CFL_DATA_PRELOADED"] = "1"
!cd {REPO} && {PY} cfl.py info

In [ ]:
# Los tests ejecutan el pipeline en GPU: si algo falla en CUDA se ve aquí y no se lanza nada.
r = subprocess.run([PY, "-m", "unittest"], cwd=REPO, capture_output=True, text=True)
lines = [l for l in r.stderr.splitlines() if "sitecustomize" not in l and "wrapt" not in l]
print("\n".join(lines[-40:] if r.returncode else lines[-4:]))
assert r.returncode == 0, "Los tests fallan en este entorno: no se lanza el plan"

## Planes

El orquestador informa cada 10 minutos de las tareas en curso y su ronda. Si un plan termina con tareas
fallidas, la celda lo dice y el registro de cada tarea queda en `resultados/<plan>/_registros/`.

In [ ]:
fallidos = []
for plan in PLANES:
    print(f"\n################ {plan} ################", flush=True)
    proc = subprocess.Popen([PY, "plan_experimentos.py", plan, "--informe-cada", "10"], cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if "sitecustomize" not in line and "wrapt" not in line:
            print(line, end="", flush=True)
    if proc.wait() != 0:
        fallidos.append(plan)
print("\nPlanes con problemas: " + ", ".join(fallidos) if fallidos else "\nTodos los planes terminaron bien.")

In [ ]:
# Informes y paquete para descargar (sin los pesos .pt, que ocupan mucho).
from pathlib import Path
for plan in PLANES:
    salida = Path(REPO) / "resultados" / Path(plan).stem
    informe = salida / "INFORME.md"
    print(f"\n================ {informe} ================")
    print(informe.read_text() if informe.is_file() else "(sin informe)")
carpetas = [f"resultados/{Path(p).stem}" for p in PLANES if (Path(REPO) / "resultados" / Path(p).stem).is_dir()]
subprocess.run(["tar", "-czf", "/kaggle/working/resultados_kaggle.tar.gz", "--exclude=*.pt", "-C", REPO, *carpetas], check=True)
print("\nPaquete: /kaggle/working/resultados_kaggle.tar.gz")
assert not fallidos, f"Planes con problemas: {fallidos}"